# Figure: measured T1 in pig hearts (Hafyane et al. 2018)

The cell below is tagged `#| label: fig-pig-hearts-output`, so `04-issues.md` embeds its output with `:::{figure} #fig-pig-hearts-output`.

In [ ]:
#| label: fig-pig-hearts-output
# Measured data: T1 of six pig hearts with the reference method and the three cardiac methods.
# Values are the averages reported in Table 1 of Hafyane et al. (2018).
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

METHODS = ["Reference (IR)", "MOLLI", "ShMOLLI", "SASHA"]
COLORS = ["gray", "royalblue", "seagreen", "darkorange"]
DATA = {   # average T1 [ms], Hafyane et al. 2018, Table 1
    "fresh hearts (weeks 1 to 3)": [1190.4, 1057.90, 989.45, 1237.4],
    "hearts fixed in formalin (weeks 4 to 6)": [758.09, 632.86, 565.30, 770.78],
}

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14,
                    subplot_titles=("Measured T1", "Difference to the reference"))
for i, (state, t1) in enumerate(DATA.items()):
    vis = (i == 0)
    ref = t1[0]
    fig.add_trace(go.Bar(x=METHODS, y=t1, marker_color=COLORS, visible=vis, showlegend=False,
                         text=[f"{v:.0f} ms" for v in t1], textposition="outside",
                         hovertemplate="%{x}: %{y:.0f} ms<extra></extra>"), row=1, col=1)
    bias = [v - ref for v in t1[1:]]
    pct = [100 * b / ref for b in bias]
    fig.add_trace(go.Bar(x=METHODS[1:], y=bias, marker_color=COLORS[1:], visible=vis,
                         showlegend=False,
                         text=[f"{b:+.0f} ms ({p:+.1f}%)" for b, p in zip(bias, pct)],
                         textposition="outside",
                         hovertemplate="%{x}: %{y:+.0f} ms<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=METHODS, y=[ref] * 4, mode="lines", visible=vis,
                             line=dict(color="black", dash="dash"), showlegend=False,
                             hoverinfo="skip"), row=1, col=1)

buttons = []
for i, state in enumerate(DATA):
    visible = [False] * (3 * len(DATA))
    visible[3 * i:3 * i + 3] = [True] * 3
    buttons.append(dict(label=state, method="update", args=[{"visible": visible}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, xanchor="left", y=1.2, yanchor="bottom")],
    height=480, template="plotly_white", margin=dict(t=110, b=60),
)
fig.update_yaxes(title_text="T1 [ms]", range=[0, 1450], row=1, col=1)
fig.update_yaxes(title_text="T1 − reference [ms]", range=[-260, 110], row=1, col=2)

# Only show values that matter: no coordinates on hover, no zooming, legend not clickable.
fig.update_layout(dragmode=False, legend=dict(itemclick=False, itemdoubleclick=False))
fig.update_xaxes(fixedrange=True)
fig.update_yaxes(fixedrange=True)
fig.update_layout(hovermode=False)   # the values are written on the bars
fig
